06_state_residual_map

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.stats import spearmanr

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict

SEED = 42
OUT_DIR = Path("outputs")
stage1 = pd.read_pickle(OUT_DIR / "stage1_all_women.pkl").reset_index(drop=True)
fl = pd.read_pickle(OUT_DIR / "feature_lists.pkl")
CAT, NUM = list(fl["FEATURES_CAT"]), list(fl["FEATURES_NUM"])
VL = pd.read_pickle(OUT_DIR / "value_labels.pkl")
SRC = {"sector": "b1q3_perv1", "social_group": "b3q4_hhv1", "religion": "b3q3_hhv1",
       "state": "state_perv1", "marital": "b4q7_perv1"}

def lab(series, var):
    m = {}
    for k, v in VL.get(SRC.get(var, var), {}).items():
        try: m[str(int(float(k)))] = v
        except Exception: m[str(k)] = v
    s = series.astype(str).str.replace(r"\.0$", "", regex=True).str.replace(r"^0+(?=\d)", "", regex=True)
    return s.map(m).fillna(s)

def pop_weight(df):
    need = ["NSS_perv1", "NSC_perv1", "mult_perv1"]
    if not set(need) <= set(df.columns):
        return pd.Series(1.0, index=df.index)
    nss, nsc, mult = [pd.to_numeric(df[c], errors="coerce") for c in need]
    return pd.Series(np.where(nss == nsc, mult / 100, mult / 200), index=df.index).fillna(0)

stage1["w"] = pop_weight(stage1)            # only ratios are used here, so the scale does not matter
stage1["state_name"] = lab(stage1["state"], "state")
print(stage1["state_name"].nunique(), "states | unlabeled codes left:",
      [s for s in stage1["state_name"].unique() if str(s).isdigit()])

In [ ]:
def make_pipe(model, cat, num):
    pre = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=30), cat),
        ("num", StandardScaler(), num)])
    return Pipeline([("pre", pre), ("model", model)])

# Replace these with gs_rf.best_params_ from 03 if you still have them
RF_PARAMS = dict(max_depth=16, min_samples_leaf=20)

def new_rf():   # NOT class-weighted, so predicted probabilities stay on the scale of the real rate
    return RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=SEED, **RF_PARAMS)

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
y, groups = stage1["lfp"], stage1["HHID"]

def oof(cat, num, model):
    return cross_val_predict(make_pipe(model, cat, num), stage1[cat + num], y,
                             groups=groups, cv=cv, method="predict_proba")[:, 1]

CAT_NO_STATE = [c for c in CAT if c != "state"]            # keeps sector (urban/rural)
CAT_NO_AREA = [c for c in CAT if c not in ("state", "sector")]   # strictest: no area variable at all

preds = {}
preds["RF, no state (main)"] = oof(CAT_NO_STATE, NUM, new_rf())
preds["LR, no state"] = oof(CAT_NO_STATE, NUM, LogisticRegression(max_iter=1000))
preds["RF, no state, no sector"] = oof(CAT_NO_AREA, NUM, new_rf())
preds["RF, WITH state (for contrast)"] = oof(CAT, NUM, new_rf())

print("Actual participation: %.3f" % y.mean())
for k, p in preds.items():
    print(f"{k:32s} mean predicted: {p.mean():.3f}")

In [ ]:
def state_gaps(p):
    d = stage1[["HHID", "state_name", "lfp", "w"]].assign(p=p)
    d["r"] = d["lfp"] - d["p"]
    rows = []
    for s, g in d.groupby("state_name"):
        n = len(g); rbar = g["r"].mean()
        hh_sums = (g["r"] - rbar).groupby(g["HHID"]).sum()          # household-clustered standard error
        se = np.sqrt((hh_sums ** 2).sum()) / n
        rows.append({"state": s, "n": n, "households": g["HHID"].nunique(),
                     "actual": g["lfp"].mean(), "predicted": g["p"].mean(), "gap": rbar, "se": se,
                     "w_actual": np.average(g["lfp"], weights=g["w"]),
                     "w_predicted": np.average(g["p"], weights=g["w"])})
    t = pd.DataFrame(rows).set_index("state")
    t["w_gap"] = t["w_actual"] - t["w_predicted"]
    t["ci_low"], t["ci_high"] = t["gap"] - 1.96 * t["se"], t["gap"] + 1.96 * t["se"]
    t["clearly_different_from_0"] = (t["ci_low"] > 0) | (t["ci_high"] < 0)
    t["small_sample"] = t["n"] < 500
    return t.sort_values("gap")

gaps = {k: state_gaps(p) for k, p in preds.items()}
main_gaps = gaps["RF, no state (main)"]
print("MAIN: actual minus predicted participation by state (negative = lower than characteristics predict)")
display(main_gaps.round(3))
main_gaps.round(4).to_csv(OUT_DIR / "state_residuals_main.csv")

In [ ]:
t = main_gaps.copy()
colors = np.where(t["gap"] < 0, "#B5483A", "#1F3A5F")
fig, ax = plt.subplots(figsize=(8, 11))
ax.barh(t.index, t["gap"], xerr=1.96 * t["se"], color=colors, alpha=0.85, ecolor="#555", capsize=2)
for i, (s, r) in enumerate(t.iterrows()):
    if r["small_sample"]:
        ax.text(0, i, " (small n)", va="center", fontsize=7, color="#555")
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("Actual minus predicted participation (95% CI)")
ax.set_title("State residuals from a model WITHOUT state\n(negative = lower participation than women's characteristics predict)")
plt.tight_layout(); plt.savefig(OUT_DIR / "fig_state_residual_map.png", dpi=150); plt.show()

print("States clearly BELOW prediction:", list(t[(t["gap"] < 0) & t["clearly_different_from_0"]].index))
print("States clearly ABOVE prediction:", list(t[(t["gap"] > 0) & t["clearly_different_from_0"]].index))

In [ ]:
ref = gaps["RF, no state (main)"]["gap"]
cmp_rows = []
for k, t_ in gaps.items():
    if k == "RF, no state (main)": continue
    common = ref.index.intersection(t_.index)
    rho = spearmanr(ref[common], t_.loc[common, "gap"])[0]
    cmp_rows.append({"alternative": k, "rank_correlation_with_main": round(rho, 3),
                     "mean_abs_gap": round(t_["gap"].abs().mean(), 4),
                     "max_abs_gap": round(t_["gap"].abs().max(), 4)})
cmp_rows.insert(0, {"alternative": "RF, no state (main)", "rank_correlation_with_main": 1.0,
                    "mean_abs_gap": round(ref.abs().mean(), 4), "max_abs_gap": round(ref.abs().max(), 4)})
display(pd.DataFrame(cmp_rows))
print("The 'WITH state' row should show gaps near zero. That is why the map cannot use a model that contains state.")

wr = pd.DataFrame({"unweighted_gap": main_gaps["gap"], "weighted_gap": main_gaps["w_gap"]})
print("Rank correlation, unweighted vs weighted gaps: %.3f" % spearmanr(wr["unweighted_gap"], wr["weighted_gap"])[0])

In [ ]:
preds["LR, WITH state"] = oof(CAT, NUM, LogisticRegression(max_iter=1000))
gaps["LR, WITH state"] = state_gaps(preds["LR, WITH state"])
print("With-state logistic regression, absolute state gap:")
print(gaps["LR, WITH state"]["gap"].abs().agg(["mean", "max"]).round(4))   # should be close to 0

main_gaps["large_gap"] = main_gaps["gap"].abs() >= 0.05
print("States with a gap of 5 points or more:", int(main_gaps["large_gap"].sum()), "of", len(main_gaps))
main_gaps.round(4).to_csv(OUT_DIR / "state_residuals_main.csv")